In [ ]:
!pip install river pandas kagglehub

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory


In [ ]:
!kaggle datasets list

You must authenticate before you can call the Kaggle API.
Follow the instructions to authenticate at: https://github.com/Kaggle/kaggle-cli/blob/main/docs/README.md#authentication


In [ ]:
!kaggle datasets download -d ealaxi/paysim1
!unzip paysim1.zip

Dataset URL: https://www.kaggle.com/datasets/ealaxi/paysim1
License(s): CC-BY-SA-4.0
Resuming from 3145728 bytes (183239833 bytes left)...
100% 178M/178M [00:00<00:00, 215MB/s]

Archive:  paysim1.zip
  inflating: PS_20174392719_1491204439457_log.csv  


In [ ]:
import pandas as pd
import numpy as np
from collections import deque
from river import stream, metrics, preprocessing, forest, drift
import random

# =========================
# LOAD DATA
# =========================
df = pd.read_csv("PS_20174392719_1491204439457_log.csv")
df = df.sort_values("step").reset_index(drop=True)

df["type_encoded"]      = df["type"].astype("category").cat.codes
df["orig_balance_diff"] = df["newbalanceOrig"] - df["oldbalanceOrg"]
df["dest_balance_diff"] = df["newbalanceDest"] - df["oldbalanceDest"]
df["amount_ratio"]      = df["amount"] / (df["oldbalanceOrg"] + 1)

X = df.drop(columns=["isFraud","isFlaggedFraud","nameOrig","nameDest","type","step"])
y = df["isFraud"]

print(f"Total rows:  {len(df):,}")
print(f"Fraud rows:  {df['isFraud'].sum():,}")
print(f"Fraud rate:  {df['isFraud'].mean()*100:.3f}%")

# =========================
# TRUE BASELINE — ROSFD paper
# ARF + ADWIN
# =========================
class ROSFD_Baseline:
    name = "ROSFD(paper)"
    def __init__(self):
        self.drift_count = 0
        self.adwin       = drift.ADWIN()
        self.model       = (preprocessing.StandardScaler() |
                           forest.ARFClassifier(n_models=10, seed=42))

    def learn_predict(self, x, y_true):
        y_proba = self.model.predict_proba_one(x).get(1, 0.0)
        y_pred  = self.model.predict_one(x)

        self.adwin.update(int(y_pred != y_true))
        if self.adwin.drift_detected:
            self.drift_count += 1

        self.model.learn_one(x, y_true)

        return y_pred, y_proba

Total rows:  6,362,620
Fraud rows:  8,213
Fraud rate:  0.129%


In [ ]:
import pandas as pd
import numpy as np
from collections import deque
from river import stream, metrics, preprocessing, forest, drift
import random

# =========================
# LOAD DATA
# =========================
df = pd.read_csv("PS_20174392719_1491204439457_log.csv")
df = df.sort_values("step").reset_index(drop=True)

df["type_encoded"]      = df["type"].astype("category").cat.codes
df["orig_balance_diff"] = df["newbalanceOrig"] - df["oldbalanceOrg"]
df["dest_balance_diff"] = df["newbalanceDest"] - df["oldbalanceDest"]
df["amount_ratio"]      = df["amount"] / (df["oldbalanceOrg"] + 1)

X = df.drop(columns=["isFraud","isFlaggedFraud","nameOrig","nameDest","type","step"])
y = df["isFraud"]

print(f"Total rows:  {len(df):,}")
print(f"Fraud rows:  {df['isFraud'].sum():,}")
print(f"Fraud rate:  {df['isFraud'].mean()*100:.3f}%")

# =========================
# H2 — ARF + ADWIN + Online SMOTE
# =========================
class H2_OSmote:
    name = "H2-OSmote"
    def __init__(self, k=5, buf_size=500, n_syn=50):
        self.k           = k
        self.n_syn       = n_syn
        self.buf         = deque(maxlen=buf_size)
        self.drift_count = 0
        self.adwin       = drift.ADWIN()
        self.model       = (preprocessing.StandardScaler() |
                           forest.ARFClassifier(n_models=10, seed=42))

    def _synthesize(self, x):
        if len(self.buf) < self.k:
            return x
        x_arr    = np.array(list(x.values()))
        neighbors= list(self.buf)
        dists    = [np.linalg.norm(x_arr - np.array(list(n.values())))
                    for n in neighbors]
        k_idx    = np.argsort(dists)[:self.k]
        neighbor = neighbors[np.random.choice(k_idx)]
        alpha    = np.random.random()
        return {f: x[f] + alpha*(neighbor[f]-x[f]) for f in x}

    def learn_predict(self, x, y_true):
        y_proba = self.model.predict_proba_one(x).get(1, 0.0)
        y_pred  = self.model.predict_one(x)

        self.adwin.update(int(y_pred != y_true))
        if self.adwin.drift_detected:
            self.drift_count += 1

        self.model.learn_one(x, y_true)
        if y_true == 1:
            self.buf.append(x)
            for _ in range(self.n_syn):
                x_syn = self._synthesize(x)
                self.model.learn_one(x_syn, 1)

        return y_pred, y_proba

Total rows:  6,362,620
Fraud rows:  8,213
Fraud rate:  0.129%


In [ ]:
import pandas as pd
from river import stream, metrics

# STREAM DATASET
dataset = stream.iter_pandas(X, y)

# INIT MODEL
model = H2_OSmote()

# metrics
auc = metrics.ROCAUC()
f1 = metrics.F1()
recall = metrics.Recall()
precision = metrics.Precision()

fraud_seen = 0
normal_seen = 0

# RUN STREAM
for i, (x_i, y_i) in enumerate(dataset):

    y_pred, y_proba = model.learn_predict(x_i, y_i)

    # update metrics
    auc.update(y_i, y_proba)
    f1.update(y_i, y_pred)
    recall.update(y_i, y_pred)
    precision.update(y_i, y_pred)

    # count classes
    if y_i == 1:
        fraud_seen += 1
    else:
        normal_seen += 1

    # logging
    if i % 100_000 == 0 and i > 0:
        print(f"\n{'='*50}")
        print(f"Step: {i:,}")
        print(f"Fraud seen: {fraud_seen:,}")
        print(f"Normal seen: {normal_seen:,}")
        print(f"{auc}")
        print(f"{f1}")
        print(f"{recall}")
        print(f"{precision}")
        print(f"Drifts:    {model.drift_count}")

# FINAL RESULT
print(f"\n{'='*50}")
print("FINAL RESULT")
print(f"{'='*50}")
print(f"{auc}")
print(f"{f1}")
print(f"{recall}")
print(f"{precision}")
print(f"Drifts:    {model.drift_count}")
print(f"Fraud seen: {fraud_seen:,}")
print(f"Normal seen: {normal_seen:,}")


Step: 100,000
Fraud seen: 116
Normal seen: 99,885
ROCAUC: 84.83%
F1: 19.69%
Recall: 37.93%
Precision: 13.29%
Drifts:    12

Step: 200,000
Fraud seen: 147
Normal seen: 199,854
ROCAUC: 83.01%
F1: 18.54%
Recall: 38.78%
Precision: 12.18%
Drifts:    18

Step: 300,000
Fraud seen: 181
Normal seen: 299,820
ROCAUC: 81.72%
F1: 17.60%
Recall: 39.78%
Precision: 11.30%
Drifts:    25

Step: 400,000
Fraud seen: 206
Normal seen: 399,795
ROCAUC: 80.78%
F1: 17.80%
Recall: 40.78%
Precision: 11.38%
Drifts:    31

Step: 500,000
Fraud seen: 233
Normal seen: 499,768
ROCAUC: 80.42%
F1: 17.27%
Recall: 40.77%
Precision: 10.96%
Drifts:    39

Step: 600,000
Fraud seen: 361
Normal seen: 599,640
ROCAUC: 84.36%
F1: 25.00%
Recall: 50.97%
Precision: 16.56%
Drifts:    47

Step: 700,000
Fraud seen: 417
Normal seen: 699,584
ROCAUC: 83.15%
F1: 24.63%
Recall: 50.36%
Precision: 16.30%
Drifts:    52

Step: 800,000
Fraud seen: 460
Normal seen: 799,541
ROCAUC: 82.29%
F1: 23.83%
Recall: 50.00%
Precision: 15.65%
Drifts:    61

